# 参考题解：多查询注意力（MQA）

实现共享单组 K/V 的 Multi-Query Attention。输入形状为 `[B, S, D]`，输出保持 `[B, S, D]`。

核心思路：查询使用多个头，而键和值只投影成一个共享头；计算注意力时依靠广播让所有查询头共享同一组 K/V。


In [ ]:
# ✅ SOLUTION

import torch
import torch.nn as nn
import math

class MultiQueryAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model must be divisible by num_heads")
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, self.head_dim)
        self.W_v = nn.Linear(d_model, self.head_dim)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x: torch.Tensor, mask=None) -> torch.Tensor:
        batch, seq_len, _ = x.shape
        q = self.W_q(x).view(batch, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).unsqueeze(1)
        v = self.W_v(x).unsqueeze(1)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).contiguous().view(batch, seq_len, self.d_model)
        return self.W_o(out)
